# 重新排序
重新排序模块根据不同的标准或不同的（例如更昂贵的）算法对搜索结果集进行重新排序。

## 命名向量
> 添加于v1.24

任何基于向量的搜索，如果集合已配置命名向量，都必须在查询中包含target向量名称。这样，Weaviate 才能找到正确的向量，并与查询向量进行比较。

In [ ]:
## V3

### 不支持


### V4
from weaviate.classes.query import MetadataQuery

reviews = client.collections.get("WineReviewNV")
response = reviews.query.near_text(
    query="a sweet German white wine",
    limit=2,
    target_vector="title_country",  # Specify the target vector for named vector collections
    return_metadata=MetadataQuery(distance=True)
)

for o in response.objects:
    print(o.properties)
    print(o.metadata.distance)

## 重新排序向量搜索结果

要对矢量搜索的结果重新排序，请配置要排序的对象属性。

In [ ]:
response = (
    client.query
    .get("JeopardyQuestion", ["question", "answer"])
    .with_near_text({
        "concepts": ["flying"]
    })
    .with_additional("rerank(property: \"answer\" query: \"floating\") { score }")
    .with_limit(10)
    .do()
)

print(json.dumps(response, indent=2))

In [ ]:
from weaviate.classes.query import Rerank, MetadataQuery

jeopardy = client.collections.get("JeopardyQuestion")

response = jeopardy.query.near_text(
    query="flying",
    limit=10,
    rerank=Rerank(
        prop="question",
        query="publication"
    ),
    return_metadata=MetadataQuery(score=True)
)

for o in response.objects:
    print(o.properties)
    print(o.metadata.score)

## 重新排名关键字搜索结果
要对关键字搜索的结果重新排序，请配置要排序的对象属性。

In [ ]:
response = (
    client.query
    .get("JeopardyQuestion", ["question", "answer"])
    .with_bm25(
      query="paper"
    )
    .with_additional("rerank(property: \"question\" query: \"publication\") { score }")
    .with_limit(10)
    .do()
)

print(json.dumps(response, indent=2))

In [ ]:
from weaviate.classes.query import Rerank, MetadataQuery

jeopardy = client.collections.get("JeopardyQuestion")

response = jeopardy.query.bm25(
    query="paper",
    limit=10,
    rerank=Rerank(
        prop="question",
        query="publication"
    ),
    return_metadata=MetadataQuery(score=True)
)

for o in response.objects:
    print(o.properties)
    print(o.metadata.rerank_score)